In [1]:
import h5py
import pandas as pd
import numpy as np
import gc

def parse_archs4_table(archs4_file, root_key, table_key):
    # see https://maayanlab.cloud/archs4/help.html for more infos on how to navigate the file
    archs4 = h5py.File(archs4_file, 'r')
    
    data = dict()
    table = archs4[root_key][table_key]
    for key in table.keys():
        column_data_piece = table[key][0]
        if isinstance(column_data_piece, np.number):
            column_data = table[key][:]
        
        else:
            column_data = table[key].asstr()[:]
        
        data[key] = column_data
    
    return data


def filter_table_data(data, retain_keys):
    retain_data = {
        key: data[key].copy() for key in retain_keys
    }
    return retain_data


def get_srx_samn_from_items(relation_items):
    srx, samn = None, None
    for item in relation_items:
        if not item:
            continue
            
        key, value = item.split(': ')
            
        if key == 'SRA':
            srx = value.split('=')[-1]
        
        if key == 'BioSample':
            samn = value.split('/')[-1]
        
    return srx, samn


def extract_srx_and_samn_accessions(relation_data):
    srx_samn_accessions = {
        key: [] for key in ['srx_accessions', 'biosample_accessions']
    }
    for relation in relation_data:
        relation_items = relation.split(',')
        srx, samn = get_srx_samn_from_items(
            relation_items
        )
        srx_samn_accessions['srx_accessions'].append(srx)
        srx_samn_accessions['biosample_accessions'].append(samn)
    
    return srx_samn_accessions
        

data = parse_archs4_table(
    'human_gene_v2.2.h5',
    'meta',
    'samples'
)

retain_keys = [
    'geo_accession', 'molecule_ch1', 'organism_ch1', 'readsaligned', 'relation', 
    'sample', 'series_id', 'singlecellprobability', 'source_name_ch1', 'title'
]

filtered_data = filter_table_data(
    data, 
    retain_keys
)

srx_samn_accessions = extract_srx_and_samn_accessions(
    filtered_data['relation']
)

srx_samn_table = pd.DataFrame().from_dict(
    srx_samn_accessions,
    orient = 'columns'
)

table = pd.DataFrame().from_dict(
    filtered_data, 
    orient = 'columns'
)

table = pd.concat(
    [table, srx_samn_table],
    axis = 1
)

del data, filtered_data, srx_samn_accessions
gc.collect()

table

,geo_accession,molecule_ch1,organism_ch1,readsaligned,relation,sample,series_id,singlecellprobability,source_name_ch1,title,srx_accessions,biosample_accessions
0,GSM1000981,total RNA,Homo sapiens,102963402,SRA: https://www.ncbi.nlm.nih.gov/sra?term=SRX...,GSM1000981,GSE29282,0.007336,Human DLBCL cel line,OCI-LY1_48hrs_mRNAseq_3x_siNT_R1,SRX185895,SAMN01163911
1,GSM1000982,total RNA,Homo sapiens,85980162,SRA: https://www.ncbi.nlm.nih.gov/sra?term=SRX...,GSM1000982,GSE29282,-0.006492,Human DLBCL cel line,OCI-LY1_48hrs_mRNAseq_3x_siNT_R2,SRX185896,SAMN01163912
2,GSM1000983,total RNA,Homo sapiens,109810687,SRA: https://www.ncbi.nlm.nih.gov/sra?term=SRX...,GSM1000983,GSE29282,-0.006492,Human DLBCL cel line,OCI-LY1_48hrs_mRNAseq_3x_siNT_R3,SRX185897,SAMN01163913
3,GSM1000984,total RNA,Homo sapiens,105304785,SRA: https://www.ncbi.nlm.nih.gov/sra?term=SRX...,GSM1000984,GSE29282,0.007336,Human DLBCL cel line,OCI-LY1_48hrs_mRNAseq_3x_siBCL6_R1,SRX185898,SAMN01163914
4,GSM1000985,total RNA,Homo sapiens,97274296,SRA: https://www.ncbi.nlm.nih.gov/sra?term=SRX...,GSM1000985,GSE29282,0.030632,Human DLBCL cel line,OCI-LY1_48hrs_mRNAseq_3x_siBCL6_R2,SRX185899,SAMN01163915
...,...,...,...,...,...,...,...,...,...,...,...,...
722420,GSM999587,polyA RNA,Homo sapiens,1225510,SRA: https://www.ncbi.nlm.nih.gov/sra?term=SRX...,GSM999587,GSE40710,0.238154,Brain cortex,Parkinson's Disease PD 13,SRX185248,SAMN01163627
722421,GSM999588,polyA RNA,Homo sapiens,2283349,SRA: https://www.ncbi.nlm.nih.gov/sra?term=SRX...,GSM999588,GSE40710,0.009471,Brain cortex,Parkinson's Disease PD 14,SRX185249,SAMN01163628
722422,GSM999589,polyA RNA,Homo sapiens,3253397,SRA: https://www.ncbi.nlm.nih.gov/sra?term=SRX...,GSM999589,GSE40710,0.083552,Brain cortex,Parkinson's Disease PD 15,SRX185250,SAMN01163629
722423,GSM999590,polyA RNA,Homo sapiens,511,SRA: https://www.ncbi.nlm.nih.gov/sra?term=SRX...,GSM999590,GSE40710,0.333697,Brain cortex,Parkinson's Disease PD 16,SRX185251,SAMN01163630


In [3]:
table.to_csv(
    'archs4_sample_metadata.tsv.gz',
    sep = '\t',
    compression = 'gzip',
    index = False
)

In [16]:
from Bio import Entrez
import multiprocessing as mp
from functools import partial
import time
import urllib
import os


Entrez.email = 'dmalzl@cemm.oeaw.ac.at'
Entrez.api_key = '746c80043a6e2ceeb01ae587f8ab1a8ace09'


def uid_from_esearch(accession, db):
    response_handle = Entrez.esearch(db = db, term = accession)
    uid_list = Entrez.read(response_handle)['IdList']
    return [[accession, uid, db] for uid in uid_list]
        

def get_uids_from_entrez(accession, db, n_retries = 5):
    success = False
    n_tries = 1
    while not success:
        if n_tries == n_retries:
            raise RuntimeError('Exceeded retries!')
            
        try:
            uid_list = uid_from_esearch(accession, db)
            success = True
        
        except urllib.error.HTTPError as e:
            print(e)
            print(f'current try: {n_tries}')
            time.sleep(10)
            n_tries += 1
        
    return uid_list


def write_to_file(filename, uid_list, filelock):
    with filelock:
        with open(filename, 'a') as outfile:
            for uid_map in uid_list:
                outfile.write(
                    '\t'.join(uid_map) + '\n'
                )
            
            outfile.flush()
        
        
def read_retrieved_accessions(filename):
    accession_map = pd.read_csv(
        filename,
        sep = '\t'
    )
    return set(accession_map.accession)


def map_accessions_to_srauid(accessions, outfilename, filelock):
    uid_list = []
    for i, row in accessions:
        srx = row.srx_accessions
        samn = row.biosample_accessions
        gsm = row.geo_accession

        if any(acc in retrieved_accessions for acc in [srx, samn, gsm]):
            continue

        database = 'sra'
        if not any([srx, samn]):
            accession = gsm

        else:
            accession = srx if srx else samn

        mapped_uids = get_uids_from_entrez(
            accession, 
            database
        )
        
        uid_list.extend(mapped_uids)
        
        if not (i + 1) % 1000:
            write_to_file(
                outfilename,
                uid_list,
                filelock
            )
            uid_list = []
            print(i + 1)
            
    # write remaining uids if there are any
    if uid_list:
        write_to_file(
            outfilename,
            uid_list,
            filelock
        )

In [17]:
outfilename = 'archs4_accession_to_uid.tsv'
if not os.path.exists(outfilename):
    retrieved_accessions = set()
    with open(outfilename, 'w') as outfile:
        outfile.write(
            'accession\tuid\tdatabase\n'
        )

else:
    retrieved_accessions = read_retrieved_accessions(outfilename)
    retrieved = table.apply(
        lambda x: any(
            x[acc] in retrieved_accessions 
            for acc in ['geo_accession', 'srx_accessions', 'biosample_accessions']
        ),
        axis = 1
    )
    table_remaining = table.loc[~retrieved, :]
    
        
table_remaining

,geo_accession,molecule_ch1,organism_ch1,readsaligned,relation,sample,series_id,singlecellprobability,source_name_ch1,title,srx_accessions,biosample_accessions
163000,GSM2994698,total RNA,Homo sapiens,1323655,BioSample: https://www.ncbi.nlm.nih.gov/biosam...,GSM2994698,GSE110499,0.753842,MM16,MM16_5,SRX3683709,SAMN08519582
163001,GSM2994699,total RNA,Homo sapiens,1884086,BioSample: https://www.ncbi.nlm.nih.gov/biosam...,GSM2994699,GSE110499,0.669480,MM16,MM16_52,SRX3683710,SAMN08519581
163002,GSM2994700,total RNA,Homo sapiens,1587034,BioSample: https://www.ncbi.nlm.nih.gov/biosam...,GSM2994700,GSE110499,0.882291,MM16,MM16_54,SRX3683711,SAMN08519580
163003,GSM2994701,total RNA,Homo sapiens,1418104,BioSample: https://www.ncbi.nlm.nih.gov/biosam...,GSM2994701,GSE110499,0.709562,MM16,MM16_57,SRX3683712,SAMN08519579
163004,GSM2994702,total RNA,Homo sapiens,1647585,BioSample: https://www.ncbi.nlm.nih.gov/biosam...,GSM2994702,GSE110499,0.796150,MM16,MM16_6,SRX3683713,SAMN08519578
...,...,...,...,...,...,...,...,...,...,...,...,...
722420,GSM999587,polyA RNA,Homo sapiens,1225510,SRA: https://www.ncbi.nlm.nih.gov/sra?term=SRX...,GSM999587,GSE40710,0.238154,Brain cortex,Parkinson's Disease PD 13,SRX185248,SAMN01163627
722421,GSM999588,polyA RNA,Homo sapiens,2283349,SRA: https://www.ncbi.nlm.nih.gov/sra?term=SRX...,GSM999588,GSE40710,0.009471,Brain cortex,Parkinson's Disease PD 14,SRX185249,SAMN01163628
722422,GSM999589,polyA RNA,Homo sapiens,3253397,SRA: https://www.ncbi.nlm.nih.gov/sra?term=SRX...,GSM999589,GSE40710,0.083552,Brain cortex,Parkinson's Disease PD 15,SRX185250,SAMN01163629
722423,GSM999590,polyA RNA,Homo sapiens,511,SRA: https://www.ncbi.nlm.nih.gov/sra?term=SRX...,GSM999590,GSE40710,0.333697,Brain cortex,Parkinson's Disease PD 16,SRX185251,SAMN01163630


In [ ]:
import itertools as it

# there seems to be no way of posting a list of accessions to the WebEnv
# so we have to do this one by one which might be extremely slow
with mp.Pool(4) as p:
    # lock needs to be a managed one otherwise passing it to the pool will fail
    filelock = mp.Manager().Lock()
    
    map_function = partial(
        map_accessions_to_srauid, 
        outfilename = outfilename, 
        filelock = filelock
    )
    
    table_chunks = it.batched(
        table_remaining.iterrows(),
        n = 50000
    )
    
    # this needs to be invoked by iterating over it
    result_iterator = p.imap(
        map_function,
        table_chunks,
        chunksize = 50000
    )
    
    for _ in result_iterator:
        continue

164000
165000
166000
167000
